<div align="center">
<img src="https://poorit.in/image.png" alt="Poorit" width="40" style="vertical-align: middle;"> <b>LPU — BACKEND & GENERATIVE AI</b>

## Day 3 — Exercise: Chunk a Document, Embed It Two Ways

**Lovely Professional University**
*Backend & Generative AI · Poorit Technologies*

</div>

---

### What You'll Do

One exercise, five steps.

```
a document  →  chunks  →  embeddings   (HuggingFace, local)
                       →  embeddings   (LiteLLM, OpenAI)
                       →  which one finds the right chunk?
```

1. **Chunk** the document with `RecursiveCharacterTextSplitter`
2. **Embed** the chunks with **HuggingFace** `sentence-transformers` — free, local, 384 numbers
3. **Embed the same chunks** with **LiteLLM** → `text-embedding-3-small` — hosted, 1536 numbers
4. **Compare** the two: do they pick the same chunk for the same question?
5. *(stretch)* break the rule on purpose and see what happens

Fill in every `___` and run the cell.

> Steps 1–2 need **no API key**. Steps 3–5 do.

---

## 1. Setup

Run these two cells. Nothing to fill in — the second one downloads the local model (~1 minute).

In [ ]:
# PROVIDED - just run this cell.
!pip install -q sentence-transformers litellm langchain-text-splitters scikit-learn

In [ ]:
# PROVIDED - just run this cell.
import os
import numpy as np
from getpass import getpass
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter
from litellm import embedding
from sklearn.metrics.pairwise import cosine_similarity

print("Imports ready")

### The document

This is what you'll chunk and search. Read it — you'll need to know whether the search
actually found the right part.

In [ ]:
# PROVIDED - just run this cell.
DOCUMENT = """
TechSolutions India was founded in 2018 by Priya Sharma and Rahul Verma. The company is
headquartered in Bhubaneswar, Odisha, with additional offices in Bangalore and Hyderabad.
It specialises in AI/ML solutions, cloud services and mobile applications, and has grown to
over 250 employees.

Priya Sharma is the CEO and co-founder. She graduated from IIT Delhi and completed her MBA at
Stanford. Rahul Verma is the CTO and co-founder; he previously worked as a Tech Lead at Google
India and specialises in Machine Learning and Cloud Architecture.

CloudAssist Pro is the flagship enterprise cloud management platform at 50,000 per month, and
includes auto-scaling, real-time monitoring and 24/7 support. SmartHR is an AI-powered HR system
at 25,000 per month. DataViz Analytics is a business intelligence platform at 15,000 per month.

Work hours are 9 AM to 6 PM, Monday to Friday, on a hybrid model with 3 days in office and 2
days remote. Employees receive 24 paid leaves and 10 sick leaves per year. Maternity leave is
26 weeks and paternity leave is 2 weeks.

Employee benefits include health insurance of 5 lakh for the employee and their family. The
learning budget is 50,000 per year per employee. Major clients include HDFC Bank, Tata Motors
and Reliance Industries.
"""

print(len(DOCUMENT), "characters")

---

## Step 1 — Chunk it

One vector cannot represent a document that talks about founders, prices, leave policy *and*
clients all at once. Cut it up first.

In [ ]:
# Hint: RecursiveCharacterTextSplitter(chunk_size=..., chunk_overlap=...)
#       then .split_text(DOCUMENT).
#       Aim for about 300 characters per chunk with about 50 of overlap.

splitter = ___(chunk_size=___, chunk_overlap=___)
chunks = splitter.___(DOCUMENT)

print(len(chunks), "chunks\n")
print("--- chunk 0 ---")
print(chunks[0])

**Check your chunks.** Every one should end on a **whole word**. If any chunk ends mid-word,
you are not using the recursive splitter.

Now change `chunk_size` to `80` and re-run. How many chunks do you get now — and is a single
80-character chunk still enough to answer a question on its own? Then put it back to `300`.

---

## Step 2 — Embed with HuggingFace (`sentence-transformers`)

`all-MiniLM-L6-v2` runs locally on the Colab CPU. Free, fast, **no API key**.

In [ ]:
# Hint: SentenceTransformer('all-MiniLM-L6-v2'), then .encode(list_of_texts).

hf_model = ___('all-MiniLM-L6-v2')
hf_vectors = hf_model.___(chunks)

print("HuggingFace vectors:", hf_vectors.shape)
print("Numbers per chunk:  ", hf_vectors.shape[1])

---

## Step 3 — Embed the *same* chunks with LiteLLM

LiteLLM gives you one function for every provider. Here it reaches OpenAI's
`text-embedding-3-small`.

> **This step needs an OpenAI key.**

In [ ]:
# PROVIDED - just run this cell.
os.environ["OPENAI_API_KEY"] = getpass("OpenAI API Key: ")

In [ ]:
# Hint: embedding(model="text-embedding-3-small", input=chunks)
#       Each vector lives at response.data[i]["embedding"].

response = ___(model="___", input=chunks)

lite_vectors = np.array([d["embedding"] for d in response.data])

print("LiteLLM vectors:", lite_vectors.shape)
print("Numbers per chunk:", lite_vectors.shape[1])
print("Tokens billed:", response.usage.total_tokens)

**Same text, same number of rows — a completely different number of columns.**
384 against 1536. Two different coordinate systems for identical input.

---

## Step 4 — Which one finds the right chunk?

Ask the same question of both sets of vectors. The question deliberately uses words that are
**not** in the document — it says *"leaves"*, not *"holidays"*.

In [ ]:
# Hint: cosine_similarity(query_vector, all_vectors)[0] gives one score per chunk.
#       .argmax() gives you the index of the best one.
#       IMPORTANT: embed the question with the SAME model as the chunks.

question = "how many holidays do staff get each year?"

# --- HuggingFace side ---
hf_q = hf_model.encode([question])
hf_scores = cosine_similarity(hf_q, hf_vectors)[0]
hf_best = hf_scores.___()

# --- LiteLLM side ---
lite_q = np.array([embedding(model="text-embedding-3-small", input=[question]).data[0]["embedding"]])
lite_scores = cosine_similarity(lite_q, ___)[0]
lite_best = lite_scores.argmax()

print("HuggingFace picked chunk", hf_best, f"(score {hf_scores[hf_best]:.3f})")
print("LiteLLM     picked chunk", lite_best, f"(score {lite_scores[lite_best]:.3f})")
print("\nSame chunk?", hf_best == lite_best)
print("\n--- what HuggingFace found ---")
print(chunks[hf_best])

Two things to notice:

1. **The words didn't match and it still worked.** The question says *holidays* and *staff*;
   the document says *leaves* and *employees*. Keyword search would have returned nothing.
2. **The two scores are different numbers**, even when both models pick the same chunk. Scores
   are model-specific — never compare a score from one model against a score from another.
   Compare **rankings**.

---

## Step 5 *(stretch)* — Break the rule on purpose

The one rule you must not break: **the same embedding model on both sides.** Try breaking it.

In [ ]:
# Hint: compare the HuggingFace query vector against the LiteLLM chunk vectors.
#       Predict what happens BEFORE you run it: an error, or silent nonsense?

mixed = cosine_similarity(hf_q, ___)      # 384-d query vs 1536-d chunks
print(mixed)

---

### ✅ What you practised

| Idea | The one-liner |
|---|---|
| **`RecursiveCharacterTextSplitter`** | breaks on paragraph → sentence → word, so chunks never end mid-word |
| **`chunk_size` / `chunk_overlap`** | ~300 chars with ~50 overlap is a sane default; overlap is insurance against bad boundaries |
| **HuggingFace `sentence-transformers`** | `SentenceTransformer(...).encode(texts)` → 384 numbers, free and local |
| **LiteLLM `embedding()`** | one function for any provider → `text-embedding-3-small` gives 1536 numbers |
| **Same model both sides** | query and chunks must share a coordinate space, or the result is meaningless |
| **Scores vs rankings** | scores don't transfer between models; rankings usually do — so use top-k, never a threshold |

**Finished early?**
1. Re-run step 1 with `chunk_size=80`, then re-run steps 2 and 4. Does the retrieved chunk still contain a usable answer?
2. Ask a question the document cannot answer — *"what is the hostel fee?"* — and look at the best score. Did either model tell you it didn't know?
3. Add `dimensions=256` to the LiteLLM call. Does the top chunk change?